# 03 · Train and evaluate (GPU)
Settings: **GPU T4 x2**, **Internet: On**, **Add Input → your dataset `rtlrl-pool`** and
**Add Input → the output of notebook 02**. For long runs use **Save Version → Save & Run All**,
which runs in the background for up to 12 hours.

Change one thing per run, and write every run in LOG.md (date, settings, result).

In [ ]:
%%capture
# Install cell copied from Unsloth's official Kaggle GRPO notebook (unslothai/notebooks,
# nb/Kaggle-Qwen2.5_(3B)-GRPO.ipynb) on 15 Oct 2026. If it fails, copy the CURRENT install
# cell from that notebook instead; the project scripts were tested with trl 0.22.2 and
# transformers 4.57.6, which this cell installs.
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.11.2', 'triton') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
try:
    import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
except Exception:
    _torch_v = ()
_torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
!uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install transformers==4.57.6
!uv pip install --no-deps trl==0.22.2

In [ ]:
%%bash
# Build Icarus Verilog 12 (VerilogEval needs exactly v12). Takes a few minutes.
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y autoconf gperf flex bison build-essential > /dev/null
rm -rf /tmp/iverilog
git clone -q --depth 1 -b v12-branch https://github.com/steveicarus/iverilog.git /tmp/iverilog
cd /tmp/iverilog
sh ./autoconf.sh > /dev/null 2>&1
./configure --prefix=/kaggle/working/iv12 > /dev/null
make -j4 > /dev/null 2>&1
make install > /dev/null
/kaggle/working/iv12/bin/iverilog -V 2>&1 | head -1

In [ ]:
# 1. Change this to your own GitHub repository (keep it public; it holds no secrets)
REPO_URL = "https://github.com/YOUR-USERNAME/YOUR-REPO.git"
POOL_DIR = "/kaggle/input/rtlrl-pool"   # the Kaggle dataset you uploaded from your Mac

import os
!git clone -q {REPO_URL} /kaggle/working/repo
!git clone -q --depth 1 https://github.com/NVlabs/verilog-eval.git /kaggle/working/verilog-eval
os.environ["RTLRL_IVERILOG"] = "/kaggle/working/iv12/bin/iverilog"
os.environ["RTLRL_VVP"] = "/kaggle/working/iv12/bin/vvp"
%cd /kaggle/working/repo
!mkdir -p data results

NB2 = "/kaggle/input/02-baselines-and-the-learnable-zone/repo"   # check the exact path in the Input panel
!cp {POOL_DIR}/heldout.jsonl {NB2}/data/zone.jsonl {NB2}/data/rft.jsonl data/
!cp {NB2}/results/*.graded.csv results/

## A · Rejection-sampling fine-tuning (RFT): train on the model's own correct answers

In [ ]:
RUN = "rft_run1"
!python scripts/train_rft.py --model unsloth/Qwen2.5-Coder-1.5B-Instruct --data data/rft.jsonl --out /kaggle/working/{RUN}

## B · GRPO on the learnable zone (use instead of A, or after A)

In [ ]:
RUN = "grpo_run1"
!RTLRL_VERDICT_LOG=/kaggle/working/{RUN}_verdicts.csv python scripts/train_grpo.py --model unsloth/Qwen2.5-Coder-1.5B-Instruct --pool data/zone.jsonl --steps 300 --out /kaggle/working/{RUN}

## Evaluate exactly like the baseline. Only the model path changes.

Set `RUN` to the run you want to evaluate (for example `RUN = "rft_run1"`).

In [ ]:
M = f"/kaggle/working/{RUN}/merged"
!python scripts/generate.py --model {M} --verilogeval /kaggle/working/verilog-eval/dataset_spec-to-rtl --n 20 --temperature 0.85 --top-p 0.95 --out results/{RUN}_ve.jsonl
!python scripts/grade.py --answers results/{RUN}_ve.jsonl --verilogeval /kaggle/working/verilog-eval/dataset_spec-to-rtl
!python scripts/compare.py results/base_ve.graded.csv results/{RUN}_ve.graded.csv
!python scripts/generate.py --model {M} --pool data/heldout.jsonl --n 20 --temperature 0.85 --top-p 0.95 --out results/{RUN}_heldout.jsonl
!python scripts/grade.py --answers results/{RUN}_heldout.jsonl --pool data/heldout.jsonl
!python scripts/compare.py results/base_heldout.graded.csv results/{RUN}_heldout.graded.csv